# OPRB benchmark: microsoft/Florence-2-base (Kaggle)

Out-of-the-box Kaggle **Run All** evaluation on the 9,000-page OPRB test set with Hugging Face checkpoint resume and periodic upload.

**Kaggle setup:**
1. Enable **GPU** (e.g., T4 x2, T4, or P100) and **Internet** in Notebook Settings.
2. Add a Kaggle Secret named `HF_TOKEN` (write token) via **Add-ons -> Secrets** to auto-create/resume a private HF dataset repo (`<username>/oprb-florence2-results`). Optionally set `HF_REPO_ID` in the environment to override the repo name.
3. Click **Run All**.

In [ ]:
%pip -q install "transformers>=4.45,<4.50" accelerate timm einops "huggingface_hub>=0.30" "jiwer>=3" pandas Pillow

In [ ]:
import json
import os
import queue
import random
import re
import shutil
import tarfile
import time
import unicodedata
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from contextlib import nullcontext
from pathlib import Path

import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from jiwer import cer, wer

MODEL_TAG = "Florence2"
MODEL_ID = "microsoft/Florence-2-base"
OPRB_REPO_ID = "kpurkayastha/OPRB"
CATEGORIES = (("text", 6000), ("nontext", 3000))
MAX_SAMPLES_PER_CATEGORY = None  # Evaluate all 9,000 pages.
RANDOM_SEED = 42
NORMALIZE_CASE = True

BATCH_SIZE = 8
NUM_BEAMS = 1  # Fast greedy decoding for Kaggle T4/P100; set to 3 for beam search.
MAX_NEW_TOKENS = 1024
PRINT_EVERY = 25
SAVE_EVERY = 25
PUSH_EVERY = 250

DATA_ROOT = Path("/tmp/oprb_test")
OUTPUT_DIR = (Path("/kaggle/working/oprb_results") if Path("/kaggle/working").exists() else Path("oprb_results")) / MODEL_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = OUTPUT_DIR / "per_image_results.csv"
SUMMARY_JSON = OUTPUT_DIR / "summary.json"

CSV_COLUMNS = [
    "category", "sample_id", "image_path", "gt_path", "occlusion_type", "occlusion_density",
    "model_id", "status", "prediction", "ground_truth", "latency_sec", "generated_tokens",
    "possibly_truncated", "cer", "wer", "token_precision", "token_recall", "token_f1",
    "gt_words", "pred_words",
]


def resolve_hf_config():
    token = (os.environ.get("HF_TOKEN") or "").strip() or None
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = (UserSecretsClient().get_secret("HF_TOKEN") or "").strip() or None
        except Exception:
            token = None
    repo_id = (os.environ.get("HF_REPO_ID") or "").strip()
    if not token:
        print("HF_TOKEN not found; HF resume/upload disabled (saving locally only).", flush=True)
        return None, "", None
    api = HfApi(token=token)
    if not repo_id:
        repo_id = f"{api.whoami()['name']}/oprb-florence2-results"
    api.create_repo(repo_id, repo_type="dataset", private=True, exist_ok=True)
    print(f"HF persistence enabled -> dataset repo: {repo_id}", flush=True)
    return token, repo_id, api


def _extract(archive_path, destination, suffix):
    destination.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, "r:gz") as archive:
        for member in archive:
            if member.isfile() and member.name.lower().endswith(suffix):
                target = destination / Path(member.name).name
                if not target.exists():
                    with archive.extractfile(member) as src, target.open("wb") as dst:
                        shutil.copyfileobj(src, dst)


def download_data():
    for category, expected in CATEGORIES:
        for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):
            folder = DATA_ROOT / category / kind
            if len(list(folder.glob(f"*{suffix}"))) < expected:
                archive = hf_hub_download(OPRB_REPO_ID, f"test/{category}/{kind}.tar.gz", repo_type="dataset")
                _extract(archive, folder, suffix)


def collect_pairs():
    pairs = []
    for category, _ in CATEGORIES:
        image_dir, gt_dir = DATA_ROOT / category / "img", DATA_ROOT / category / "gt"
        names = sorted({p.stem for p in image_dir.glob("*.jpg")} & {p.stem for p in gt_dir.glob("*.txt")})
        if MAX_SAMPLES_PER_CATEGORY is not None:
            names = sorted(random.Random(RANDOM_SEED).sample(names, min(len(names), MAX_SAMPLES_PER_CATEGORY)))
        for stem in names:
            parts = stem.split("__", 2)
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(image_dir / f"{stem}.jpg"),
                "gt_path": str(gt_dir / f"{stem}.txt"),
                "occlusion_type": parts[0],
                "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
            })
    if not pairs:
        raise RuntimeError(f"No paired OPRB files found under {DATA_ROOT}")
    return pairs


def gt_text(path):
    with open(path, encoding="utf-8-sig", errors="replace") as f:
        return " ".join(
            line.rstrip("\r\n").split("\t")[0].strip()
            for line in f
            if "\t" in line and line.split("\t", 1)[0].strip()
        )


def norm(s):
    s = re.sub(r"\s+", " ", unicodedata.normalize("NFKC", s)).strip()
    return s.casefold() if NORMALIZE_CASE else s


def metrics(ref, hyp):
    ref, hyp = norm(ref), norm(hyp)
    if not ref:
        raise ValueError("empty ground truth")
    a, b = Counter(ref.split()), Counter(hyp.split())
    matched = sum((a & b).values())
    p = matched / sum(b.values()) if b else 0.0
    r = matched / sum(a.values()) if a else 0.0
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": 2 * p * r / (p + r) if p + r else 0.0,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }


def save_csv_atomic(rows, path):
    tmp = path.with_suffix(".csv.tmp")
    if rows:
        df = pd.DataFrame(rows)
        extra = [c for c in df.columns if c not in CSV_COLUMNS]
        df = df.reindex(columns=CSV_COLUMNS + extra)
    else:
        df = pd.DataFrame(columns=CSV_COLUMNS)
    df.to_csv(tmp, index=False)
    tmp.replace(path)


HF_TOKEN, HF_REPO_ID, hf_api = resolve_hf_config()
download_data()
pairs = collect_pairs()
print(f"Prepared {len(pairs):,} paired OPRB samples | output: {OUTPUT_DIR}", flush=True)


In [ ]:
import torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor

has_cuda = torch.cuda.is_available()
n_devices = max(1, torch.cuda.device_count()) if has_cuda else 1
DTYPE = torch.float16 if has_cuda else torch.float32
devices = [f"cuda:{i}" for i in range(n_devices)] if has_cuda else ["cpu"]
device_names = [torch.cuda.get_device_name(i) for i in range(n_devices)] if has_cuda else ["CPU"]

# Load one independent (processor, model) replica per device for thread-safe multi-GPU inference in Jupyter.
workers = []
for dev, dev_name in zip(devices, device_names):
    proc = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=DTYPE,
        attn_implementation="eager",
    ).to(dev).eval()
    workers.append({"device": dev, "device_name": dev_name, "processor": proc, "model": mdl})

device_summary = f"{len(workers)}x {device_names[0]}" if has_cuda and len(set(device_names)) == 1 else ", ".join(device_names)
print(
    f"Loaded {MODEL_ID} | workers={len(workers)} ({device_summary}) | dtype={DTYPE} | bs={BATCH_SIZE}/worker | beams={NUM_BEAMS}",
    flush=True,
)


In [ ]:
def transcribe_batch(worker, paths):
    dev = worker["device"]
    proc = worker["processor"]
    mdl = worker["model"]
    start = time.perf_counter()
    images = []
    for p in paths:
        with Image.open(p) as img:
            images.append(img.convert("RGB"))
    prompts = ["<OCR>"] * len(images)
    inputs = proc(text=prompts, images=images, return_tensors="pt", padding=True)
    # Cast floating tensors (pixel_values) to DTYPE so float16 CUDA weights do not mismatch float32 inputs.
    inputs = {
        k: (v.to(device=dev, dtype=DTYPE) if torch.is_floating_point(v) else v.to(dev))
        for k, v in inputs.items()
    }
    ctx = torch.cuda.device(dev) if dev.startswith("cuda") else nullcontext()
    with ctx, torch.inference_mode():
        ids = mdl.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=NUM_BEAMS, do_sample=False)
    raw_list = proc.batch_decode(ids, skip_special_tokens=False)
    pad_id = proc.tokenizer.pad_token_id
    per_item_latency = (time.perf_counter() - start) / len(images)
    outputs = []
    for k, image in enumerate(images):
        parsed = proc.post_process_generation(raw_list[k], task="<OCR>", image_size=image.size)
        text = parsed.get("<OCR>", parsed) if isinstance(parsed, dict) else str(parsed)
        gen_tokens = int((ids[k] != pad_id).sum().item()) if pad_id is not None else int(ids.shape[-1])
        outputs.append((str(text).strip(), per_item_latency, gen_tokens, gen_tokens >= MAX_NEW_TOKENS))
    return outputs


In [ ]:
try:
    _ = display
except NameError:
    display = print


def push_files_to_hf(paths, message):
    if not (hf_api and HF_REPO_ID):
        return
    for path in paths:
        if not path.exists():
            continue
        try:
            hf_api.upload_file(
                path_or_fileobj=str(path),
                path_in_repo=path.name,
                repo_id=HF_REPO_ID,
                repo_type="dataset",
                token=HF_TOKEN,
                commit_message=message,
            )
            print(f"HF upload complete ({path.name}): {message}", flush=True)
        except Exception as exc:
            print(f"HF upload warning ({path.name}): {exc}", flush=True)


if (not RESULTS_CSV.exists() or RESULTS_CSV.stat().st_size == 0) and HF_REPO_ID and HF_TOKEN:
    try:
        hf_hub_download(
            HF_REPO_ID,
            "per_image_results.csv",
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir=str(OUTPUT_DIR),
        )
        print(f"Resumed checkpoint from HF repo {HF_REPO_ID}", flush=True)
    except Exception as exc:
        print(f"No remote checkpoint in {HF_REPO_ID} yet ({type(exc).__name__}); starting fresh.", flush=True)

valid_ids = {p["sample_id"] for p in pairs}
old = pd.read_csv(RESULTS_CSV).to_dict("records") if RESULTS_CSV.exists() and RESULTS_CSV.stat().st_size > 0 else []
rows = []
finished = set()
for r in old:
    sid = r.get("sample_id")
    if sid in valid_ids and r.get("status") == "ok" and sid not in finished:
        rows.append(r)
        finished.add(sid)

remaining = [p for p in pairs if p["sample_id"] not in finished]
print(f"Resume state: {len(finished):,}/{len(pairs):,} completed; remaining: {len(remaining):,}", flush=True)

run_model_id = f"{MODEL_ID} (bs={BATCH_SIZE}, beams={NUM_BEAMS})"
t0, done, n_err = time.perf_counter(), 0, 0
last_printed, last_saved, last_pushed = 0, 0, 0
total_rem = len(remaining)

if total_rem > 0:
    print(
        f"Starting evaluation: {total_rem:,} samples across {len(workers)} worker(s) ({device_summary}) | bs={BATCH_SIZE}/worker",
        flush=True,
    )
    in_q = queue.Queue()
    out_q = queue.Queue()
    for offset in range(0, total_rem, BATCH_SIZE):
        in_q.put(remaining[offset : offset + BATCH_SIZE])

    def run_worker(worker):
        dev = worker["device"]
        try:
            while True:
                try:
                    batch = in_q.get_nowait()
                except queue.Empty:
                    break
                try:
                    preds = transcribe_batch(worker, [s["image_path"] for s in batch])
                except Exception:
                    if dev.startswith("cuda"):
                        with torch.cuda.device(dev):
                            torch.cuda.empty_cache()
                    preds = []
                    for s in batch:
                        try:
                            preds.append(transcribe_batch(worker, [s["image_path"]])[0])
                        except Exception as exc:
                            preds.append(exc)
                out_q.put((batch, preds))
        finally:
            out_q.put(None)

    with ThreadPoolExecutor(max_workers=len(workers)) as pool:
        futures = [pool.submit(run_worker, w) for w in workers]
        active_workers = len(workers)
        while active_workers > 0:
            item = out_q.get()
            if item is None:
                active_workers -= 1
                continue
            batch, preds = item
            for sample, pred in zip(batch, preds):
                row = {**sample, "model_id": run_model_id}
                if isinstance(pred, Exception):
                    row.update({"status": "error", "error": repr(pred)})
                    n_err += 1
                else:
                    prediction, latency, generated, truncated = pred
                    try:
                        reference = gt_text(sample["gt_path"])
                        row.update({
                            "status": "ok",
                            "prediction": prediction,
                            "ground_truth": reference,
                            "latency_sec": latency,
                            "generated_tokens": generated,
                            "possibly_truncated": truncated,
                            **metrics(reference, prediction),
                        })
                        finished.add(sample["sample_id"])
                    except Exception as exc:
                        row.update({"status": "error", "error": repr(exc)})
                        n_err += 1
                rows.append(row)
                done += 1

            if done - last_saved >= SAVE_EVERY:
                save_csv_atomic(rows, RESULTS_CSV)
                last_saved = done
            if done - last_pushed >= PUSH_EVERY:
                save_csv_atomic(rows, RESULTS_CSV)
                last_saved = done
                push_files_to_hf([RESULTS_CSV], f"checkpoint {len(finished)}/{len(pairs)}")
                last_pushed = done
            if done - last_printed >= PRINT_EVERY or done == total_rem:
                elapsed = time.perf_counter() - t0
                rate = done / elapsed if elapsed else 0.0
                eta_min = (total_rem - done) / rate / 60 if rate else 0.0
                pct = 100.0 * done / total_rem
                print(
                    f"[Progress] {done:,}/{total_rem:,} ({pct:.1f}%) | ok={len(finished):,}/{len(pairs):,} | errors={n_err:,} | rate={rate:.2f} img/s | ETA={eta_min:.1f} min",
                    flush=True,
                )
                last_printed = done
        for fut in futures:
            fut.result()

save_csv_atomic(rows, RESULTS_CSV)
elapsed_total = time.perf_counter() - t0
print(f"Evaluation complete: {len(finished):,}/{len(pairs):,} ok | errors={n_err:,} | elapsed={elapsed_total:.1f}s", flush=True)
results = pd.DataFrame(rows)
ok = results.loc[results["status"] == "ok"] if len(results) else results
summary = {
    "model": run_model_id,
    "dataset": "kpurkayastha/OPRB test",
    "total_expected": len(pairs),
    "n_success": int(len(ok)),
    "n_errors": int(len(results) - len(ok)),
    "mean_latency_sec": float(ok["latency_sec"].mean()) if len(ok) else None,
    "mean_cer": float(ok["cer"].mean()) if len(ok) else None,
    "mean_wer": float(ok["wer"].mean()) if len(ok) else None,
    "mean_token_f1": float(ok["token_f1"].mean()) if len(ok) else None,
    "batch_size": BATCH_SIZE,
    "num_beams": NUM_BEAMS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "device": device_summary,
    "num_workers": len(workers),
    "dtype": str(DTYPE),
    "runtime_sec_this_run": elapsed_total,
}
SUMMARY_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")
push_files_to_hf([RESULTS_CSV, SUMMARY_JSON], f"final {summary['n_success']}/{summary['total_expected']}")
print(json.dumps(summary, indent=2), flush=True)
display(ok[["category", "cer", "wer", "token_f1", "latency_sec"]].describe(include="all") if len(ok) else results)
